In [4]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline, AutoModelForCausalLM


from inference_wrapper import predict_missing_pico
from prompt_generator import build_pico_prompt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("Loading BioBERT Encoder from Google Drive...")
model_path = "##"

encoder_tokenizer = AutoTokenizer.from_pretrained(model_path)
encoder_model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

print("Loading BioMistral 7B Decoder...")
llm_model_name = "BioMistral/BioMistral-7B"

mistral_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
mistral_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.float16, 
    device_map="auto",
    use_safetensors=False
)

generator = pipeline(
    "text-generation",
    model=mistral_model,
    tokenizer=mistral_tokenizer
)

print("\n" + "="*40)
print("RUNNING METHOD 1 PIPELINE")
print("="*40)

sample_abstract = (
    "We conducted a randomized controlled trial to evaluate the efficacy of a new "
    "cognitive behavioral therapy approach. The primary outcome measured was the "
    "reduction in severe anxiety symptoms after six months."
)

print(f"Input Abstract:\n{sample_abstract}\n")

prediction = predict_missing_pico(sample_abstract, encoder_model, encoder_tokenizer, device)
print(f"1. BioBERT Diagnosis: {prediction}")

# Step B: Handshake generates the strict prompt
prompt = build_pico_prompt(sample_abstract, prediction)
print("2. Prompt Generated Successfully.")

print("3. BioMistral Generating Response...\n")

# Step C: Decoder writes the question
llm_output = generator(
    prompt,
    max_new_tokens=75,
    do_sample=True,
    temperature=0.3,
    return_full_text=False
)

final_result = llm_output[0]['generated_text'].strip()

print("========================================")
print("FINAL OUTPUT TO AUTHORS:")
print("========================================")
print(final_result)

Loading BioBERT Encoder from Google Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading BioMistral 7B Decoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

...

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does no

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


RUNNING METHOD 1 PIPELINE
Input Abstract:
We conducted a randomized controlled trial to evaluate the efficacy of a new cognitive behavioral therapy approach. The primary outcome measured was the reduction in severe anxiety symptoms after six months.



Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=75) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. BioBERT Diagnosis: Missing I
2. Prompt Generated Successfully.
3. BioMistral Generating Response...

FINAL OUTPUT TO AUTHORS:
What intervention was used in the study?


In [ ]:
from Bio import Entrez
import pandas as pd


Entrez.email = "veranki2@illinois.edu"

print("1. Searching PubMed for fresh, unstructured clinical trials (2025-2026)...")


search_query = '"clinical trial"[Publication Type] AND ("2025/01/01"[Date - Publication] : "2026/12/31"[Date - Publication]) NOT "structured abstract"[Filter]'


handle = Entrez.esearch(db="pubmed", term=search_query, retmax=20)
record = Entrez.read(handle)
handle.close()

id_list = record["IdList"]
print(f"Found {len(id_list)} recent IDs. Fetching the text...")

abstracts = []
if id_list:
    handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    for pubmed_article in records['PubmedArticle']:
        try:
            article = pubmed_article['MedlineCitation']['Article']
            if 'Abstract' in article:
                abstract_texts = article['Abstract']['AbstractText']
                # Join text to ensure it's a single unstructured block
                full_abstract = " ".join([str(text) for text in abstract_texts])

                # Basic sanity filter: must be a decent length, avoid obvious hardcoded headers
                if len(full_abstract) > 400 and "BACKGROUND:" not in full_abstract.upper():
                    abstracts.append(full_abstract)
        except Exception as e:
            continue

df_test = pd.DataFrame({"abstract": abstracts})
df_test.to_csv("/content/drive/MyDrive/Vasi/fresh_unstructured_test_data.csv", index=False)
print(f"Successfully saved {len(df_test)} fresh, unstructured abstracts to Google Drive!\n")




1. Searching PubMed for fresh, unstructured clinical trials (2025-2026)...
Found 20 recent IDs. Fetching the text...
Successfully saved 20 fresh, unstructured abstracts to Google Drive!



In [13]:
import pandas as pd


df_test = pd.read_csv("/content/drive/MyDrive/Vasi/fresh_unstructured_test_data.csv")


n_samples = min(10, len(df_test))
df_sampled = df_test.sample(n=n_samples, random_state=42).reset_index(drop=True)


results_data = []

print(f"Starting pipeline for {n_samples} random abstracts...\n")

for i, test_abstract in enumerate(df_sampled['abstract']):
    print(f"--- Processing Abstract {i+1}/{n_samples} ---")

   
    prediction = predict_missing_pico(test_abstract, encoder_model, encoder_tokenizer, device)
    print(f"BioBERT Diagnosis: {prediction}")

  
    prompt = build_pico_prompt(test_abstract, prediction)

   
    llm_output = generator(
        prompt,
        do_sample=True,
        temperature=0.3,
        return_full_text=False 
    )

    generated_text = llm_output[0]['generated_text'].strip()

    
    results_data.append({
        "Abstract": test_abstract,
        "Missing_Label": prediction,
        "Prompt_Asked": prompt,
        "Generated_Output": generated_text
    })


df_results = pd.DataFrame(results_data)


output_csv_path = "##"
df_results.to_csv(output_csv_path, index=False)
df_results.head()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting pipeline for 10 random abstracts...

--- Processing Abstract 1/10 ---
BioBERT Diagnosis: Missing P


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Processing Abstract 2/10 ---
BioBERT Diagnosis: Missing P


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Processing Abstract 9/10 ---
BioBERT Diagnosis: Complete


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Processing Abstract 10/10 ---
BioBERT Diagnosis: Missing P

Pipeline Complete! Saved 10 results to: /content/drive/MyDrive/Vasi/pipeline_evaluation_results.csv


,Abstract,Missing_Label,Prompt_Asked,Generated_Output
0,Key clinical features of neonatal acute respir...,Missing P,You are an expert clinical data extractor and ...,What is the gestational age of the study popul...
1,There is a paucity of data to guide the manage...,Missing P,You are an expert clinical data extractor and ...,What is the specific patient population of thi...
2,Intravenous crystalloid fluid infusion is a ma...,Complete,You are an expert clinical data extractor and ...,Patients undergoing lumbar spinal surgery who ...
3,With the rapid global uptake of glucagon-like ...,Missing P,You are an expert clinical data extractor and ...,Please clarify if the authors are asking about...
4,To compare the efficacy and safety of intravit...,Complete,You are an expert clinical data extractor and ...,Patients with diabetic retinopathy or retinal ...
